---
title: "Download Kotliarov 2020 Data"
subtitle: "scRNA-seq and CITE-seq datasets"
author: "Dr <strong>Pedro Henrique da Costa Avelar</strong><br> (edited by Dr Jon Cardoso-Silva)"
date: 2026
---

Downloads the [Kotliarov et al. (2020)](https://doi.org/10.1038/s41591-020-0769-8) scRNA-seq and [CITE-seq](https://doi.org/10.1038/nmeth.4380) datasets from Google Drive into the local data directory. Skips any file that already exists locally. Ends with a basic sanity check on cell counts and cell-type labels.

⚙️ **Imports and Constants**

In [ ]:
import sys
import gdown
import scanpy as sc

from pathlib import Path
from IPython.display import Markdown, display

# Notebook is in experiments/kotliarov2020/, so project root is two levels up.
PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import DATA_PATH, SCCITE_FNAME, SCCITE_LINK, SCRNA_FNAME, SCRNA_LINK

DATASETS = [
    ("rna", SCRNA_FNAME, SCRNA_LINK),
    ("cite", SCCITE_FNAME, SCCITE_LINK),
]

# Section 1: Download

In [ ]:
display(Markdown(f"**Data directory:** `{DATA_PATH}`"))
for data_type, fname, _ in DATASETS:
    display(Markdown(f"- {data_type}: `{fname}`"))

data = {}

for data_type, fname, link in DATASETS:
    fpath = DATA_PATH / fname
    if fpath.exists():
        display(Markdown(f"**{data_type}:** found at `{fpath}`, loading"))
        data[data_type] = sc.read_h5ad(fpath)
    else:
        display(Markdown(f"**{data_type}:** not found, downloading to `{fpath}`"))
        fpath.parent.mkdir(parents=True, exist_ok=True)
        gdown.download(link, str(fpath))
        data[data_type] = sc.read_h5ad(fpath)

display(Markdown("All datasets loaded."))

# Section 2: Sanity Check

In [ ]:
for data_type, adata in data.items():
    display(Markdown(f"**{data_type}:** {adata.n_obs:,} cells x {adata.n_vars:,} features"))
    cell_types = sorted(adata.obs["cell_type"].unique())
    display(Markdown(f"- Cell types ({len(cell_types)}): {', '.join(cell_types)}"))